<a href="https://colab.research.google.com/github/team-telnyx/telnyx-code-examples/blob/main/cookbooks/inference/04_Transcription_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build a Transcription and Translation Pipeline with Telnyx AI

In this cookbook, you'll build a simple pipeline that takes an audio file, transcribes it with Telnyx Speech-to-Text, and translates the transcript to English with Telnyx Inference.

This notebook uses:

- Speech-to-Text model: `openai/whisper-large-v3-turbo`
- Translation model: `moonshotai/Kimi-K2.5`
- Transcription API: `https://developers.telnyx.com/docs/voice/stt/rest-api`
- Transcription models: `https://developers.telnyx.com/docs/voice/stt/models`
- Chat completions endpoint: `https://api.telnyx.com/v2/ai/chat/completions`

In [ ]:
!pip install requests

### Set up API Key

In [ ]:
import os
from getpass import getpass

os.environ["TELNYX_API_KEY"] = getpass("Enter your Telnyx API key: ")

### Set Model Names

In [ ]:
STT_MODEL = "openai/whisper-large-v3-turbo"
TRANSLATION_MODEL = "moonshotai/Kimi-K2.5"
# For this transcription + translation cookbook, if we use direct requests calls, use:
TELNYX_API_BASE = "https://api.telnyx.com"

## Upload an audio file

Upload an audio file that contains speech you want to transcribe and translate.

Supported formats include `.mp3`, `.mp4`, `.m4a`, `.wav`, `.flac`, `.ogg`, and `.webm`.

In [ ]:
from google.colab import files
from pathlib import Path
from IPython.display import Audio, display

uploaded = files.upload()

uploaded_audio_path = Path(next(iter(uploaded.keys())))

print(f"Uploaded file: {uploaded_audio_path}")
print(f"File size: {uploaded_audio_path.stat().st_size:,} bytes")

display(Audio(str(uploaded_audio_path)))

In [ ]:
audio_path = uploaded_audio_path

## Transcribe the audio

Send the audio file to the Telnyx Speech-to-Text API.

This example uses `openai/whisper-large-v3-turbo` for multilingual transcription.

In [ ]:
import os
import requests
import mimetypes
from pprint import pprint

TRANSCRIPTION_URL = f"{TELNYX_API_BASE}/v2/ai/audio/transcriptions"

headers = {
    "Authorization": f"Bearer {os.environ['TELNYX_API_KEY']}"
}

mime_type = mimetypes.guess_type(audio_path.name)[0] or "application/octet-stream"

with open(audio_path, "rb") as audio_file:
    files_payload = {
        "file": (audio_path.name, audio_file, mime_type)
    }

    data = {
        "model": STT_MODEL,
        "response_format": "verbose_json"
    }

    transcription_response = requests.post(
        TRANSCRIPTION_URL,
        headers=headers,
        files=files_payload,
        data=data,
        timeout=300,
    )

print("Status:", transcription_response.status_code)
print("Response:", transcription_response.text)

transcription_response.raise_for_status()
transcription = transcription_response.json()

print("Full transcription response:")
pprint(transcription)

In [ ]:
print("Transcript text:")
transcript_text = transcription.get("text") or transcription.get("transcript")

print(transcript_text)

## Translate the transcript

Now send the transcript to a Telnyx-hosted LLM and ask it to translate the text into English.

This example uses `moonshotai/Kimi-K2.5` for translation.

In [ ]:
CHAT_COMPLETIONS_URL = f"{TELNYX_API_BASE}/v2/ai/chat/completions"

translation_payload = {
    "model": TRANSLATION_MODEL,
    "messages": [
        {
            "role": "system",
            "content": (
                "Translate the provided transcript faithfully into English. "
                "Do not summarize, omit, redact, or add commentary."
            ),
        },
        {
            "role": "user",
            "content": transcript_text,
        },
    ],
    "temperature": 0,
}

translation_response = requests.post(
    CHAT_COMPLETIONS_URL,
    headers={
        "Authorization": f"Bearer {os.environ['TELNYX_API_KEY']}",
        "Content-Type": "application/json",
    },
    json=translation_payload,
    timeout=300,
)

print("Status:", translation_response.status_code)
print("Response:", translation_response.text)

translation_response.raise_for_status()
translation = translation_response.json()

In [ ]:
translated_text = translation["choices"][0]["message"]["content"]

print(translated_text)

## Save the transcript and translation

The reference pipeline writes separate files for the source transcript, English translation, and metadata.

This notebook saves the same three outputs locally in Colab.

In [ ]:
import json
from datetime import datetime, timezone

source_output_path = Path("telnyx-spanish-support-sample.source.txt")
english_output_path = Path("telnyx-spanish-support-sample.en.txt")
metadata_output_path = Path("telnyx-spanish-support-sample.meta.json")

metadata = {
    "source_audio_file": str(audio_path),
    "source_transcript_file": str(source_output_path),
    "english_translation_file": str(english_output_path),
    "stt_model": STT_MODEL,
    "translation_model": TRANSLATION_MODEL,
    "completed_at": datetime.now(timezone.utc).isoformat(),
}

source_output_path.write_text(transcript_text, encoding="utf-8")
english_output_path.write_text(translated_text, encoding="utf-8")
metadata_output_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved files:")
print(source_output_path)
print(english_output_path)
print(metadata_output_path)

In [ ]:
# Optionally you can download these files from Google colab as follows":
from google.colab import files

files.download(source_output_path)
files.download(english_output_path)
files.download(metadata_output_path)

## Explore More
*   Want to build more with Telnyx Inference? Try the Telnyx Mission Control Portal: https://portal.telnyx.com/
*   Browse Telnyx Inference docs: https://developers.telnyx.com/docs/inference
*   Explore Telnyx examples on GitHub: https://github.com/team-telnyx/telnyx-code-examples

